In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path) #Reading of dataset



In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:

df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df_droppedTarget= df.drop('Order_ID',axis=1).copy() #Created a copy of dataset before dropping if I want to use the original dataset with target column later

display(df_droppedTarget)



In [ ]:
# Task 2: Write your code here:
df_droppedTarget.info()
def check_missing_values(df):
  missing_values = df.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_droppedTarget) #Alot of missing values in different columns


missing_cols = ['Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs','Delivery_Time']

cat_cols = df_droppedTarget[missing_cols].select_dtypes(include='object').columns
num_cols = df_droppedTarget[missing_cols].select_dtypes(include='number').columns
display(num_cols)
for i in cat_cols:
  mode = df_droppedTarget[i].mode()
  df_droppedTarget[i].fillna(mode,inplace=True)
for i in num_cols:
  mean = df_droppedTarget[i].mean()
  df_droppedTarget[i].fillna(mean,inplace=True)

df_droppedTarget.info()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_droppedTarget)


In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import LabelEncoder
df_droppedTarget.info()




In [ ]:
cat_cols = df_droppedTarget.select_dtypes(include='object').columns
display(cat_cols)


In [ ]:
label_encoders = {}
display(df_droppedTarget)

for col in cat_cols:
  le = LabelEncoder()
  df_droppedTarget[col] = le.fit_transform(df_droppedTarget[col])
  label_encoders[col] = le
display(df_droppedTarget)

In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import StandardScaler

numerical_cols = df_droppedTarget.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time") # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_droppedTarget[numerical_cols] = scaler.fit_transform(df_droppedTarget[numerical_cols])
df_droppedTarget.head()

In [ ]:
# Task 6: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")


#There is some skewness of the target data

In [ ]:
# Task 1: Write your code here:
X = df_droppedTarget.drop("Delivery_Time", axis=1).astype(float)
y = df_droppedTarget['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso

Rf = RandomForestRegressor(n_estimators=200)
Ridge = Ridge(alpha=1.0, max_iter=10000)
Lasso = Lasso(alpha=1.0,  max_iter=10000)
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled

kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_mae = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]



    # Train
  Rf.fit(X_train, y_train)
  Ridge.fit(X_train,y_train)
  Lasso.fit(X_train,y_train)
    # Predict
  y_pred = Rf.predict(X_test)

    # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)

    # Store results
  lr_mae.append(mae)

average_losses = np.mean(lr_mae, axis=0)


print(f"average loss: {average_losses}")

In [ ]:
# Task 1: Write your code here:



coeffs = {}

coeffs['Lasso'] = Lasso.coef_
coeffs['Ridge'] = Ridge.coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()




In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: